# Coding Learning Check: A Network's Forward Pass

**CS 463 · Due Wednesday, Nov 18, 11:59 PM in Canvas.** This takes the place of Wednesday's Learning Check.

In [From One Neuron to a Network](ch_6_and_10_02_neurons_to_networks.ipynb), you followed a forward pass through a small network that classifies XOR. Here you write that forward pass yourself, then test it:

1. **The step function.**
2. **`forward`:** the hidden layer, `X @ W1.T + b1`, then the activation.
3. **Test points:** run the step network on points you choose.
4. **Sigmoid:** run the same network with sigmoid hidden units, and take a guess at why it fails.

An optional extension at the end shows what happens with no activation at all.

### How this works

- **Write your code where `# YOUR CODE HERE` appears.** The comments tell you what to write; most tasks are one or two lines.
- **Stuck? Open the hints** under each task. They show the tools you need on made-up examples.
- **Run the check cell after each task.** It prints ✓ when your code is right and ✗ when something's off.
- **Task 4 also asks for a short written answer.** Replace `YOUR ANSWER HERE` in the **4 Answer:** cell.
- **To submit:** Restart and Run All, make sure every check prints ✓, then upload this notebook to Canvas.

| Task | What you turn in |
|---|---|
| 1 | Code |
| 2 | Code |
| 3 | Code |
| 4 | Code + Written |
| Optional | Code + Written (not graded) |

## Setup

Only NumPy is needed. `check` prints ✓ or ✗.

In [ ]:
import numpy as np

def check(what, ok):
    print(('✓ ' if ok else '✗ ') + what)

## Task 1: The step function

From [Section 1](ch_6_and_10_02_neurons_to_networks.ipynb#section-1) of the demo, a neuron's output is $g(z)$, where $g$ is its activation function. This assignment uses two:

$$
\text{sigmoid}(z) = \frac{1}{1 + e^{-z}}, \qquad
\text{step}(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{otherwise} \end{cases}.
$$

**Worked example.** `sigmoid` works on a whole array at once, entry by entry, because NumPy's `np.exp`, `+`, and `/` do.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print(sigmoid(np.array([-2.0, 0.0, 3.0])).round(3))

**1 · Code.** Write `step`. It should take an array `z` and return an array of the same shape, like `sigmoid`.


<details>
<summary>Hint: comparisons work entry by entry</summary>

A comparison between an array and a number compares **every entry** with that number, and returns an array of `True`/`False`, the same shape:

```python
a = np.array([4, -1, 0, 7])
print(a >= 4)    # [ True False False  True]
print(a == 0)    # [False False  True False]
```

`.astype(...)` returns a copy converted to another type, and `True`/`False` become `1`/`0`. You can chain it right onto a comparison:

```python
print((a >= 4).astype(float))   # [1. 0. 0. 1.]
print((a == 0).astype(int))     # [0 0 1 0]
```

The parentheses matter: `a >= 4.astype(float)` would try to call `.astype` on the number `4`.

</details>

In [ ]:
# 1
def step(z):
    "1.0 where z > 0, and 0.0 elsewhere."
    ### YOUR CODE HERE


In [ ]:
z_test = np.array([-2.0, 0.0, 3.0])
check('step(z_test) is [0, 0, 1]', np.array_equal(step(z_test), [0, 0, 1]))
check('step works on a 2D array', np.array_equal(step(np.array([[-1.0, 2.0], [0.5, 0.0]])), [[0, 1], [1, 0]]))

## Task 2: The forward pass

A layer's weights form a matrix `W1` with **one row per unit**. For every data point at once (the rows of `X`), the layer's scores are `Z1 = X @ W1.T + b1`: one row per data point, one column per unit. The `.T` turns `W1`'s rows (the units' weight vectors) into columns, so every data point is dotted with every unit's weights, and `b1` adds each unit's bias to every row. (See the demo's [notation section](ch_6_and_10_02_neurons_to_networks.ipynb#notation), and Lab 6, Part 8.) Then the activation applies to every entry.

**Worked example.** The demo's transpose example: unit 1's weights are $(1, 2)$, and unit 2's are $(-1, 1)$.

In [ ]:
W_ex = np.array([[ 1.0, 2.0],     # unit 1's weights
                 [-1.0, 1.0]])    # unit 2's weights
b_ex = np.array([0.0, 0.0])
X_ex = np.array([[0.0, 0.0],
                 [1.0, 0.0],
                 [2.0, 1.0]])

Z_ex = X_ex @ W_ex.T + b_ex
print(Z_ex)                       # row 3 is (4, -1), as in the demo
print('shape:', Z_ex.shape)       # (3 points, 2 units)

**2 · Code.** Finish `forward`, which runs a network with one hidden layer on every row of `X` and returns the output scores. Write the hidden layer's two lines: its scores `Z1`, and its outputs `H`. `g` is the hidden layer's activation, passed in as a function, so you can run the same network with `step` or `sigmoid`.

The output layer's two lines are given. They follow the same pattern as the hidden layer, with `H` as the input: `W2` has one row per output unit, and there's just one, so `Z2 = H @ W2.T + b2` has shape (number of data points, 1). `forward` returns that one column, `Z2[:, 0]`. (The demo's [notation section](ch_6_and_10_02_neurons_to_networks.ipynb#notation) explains this.)


<details>
<summary>Hint for <code>Z1</code></summary>

Follow the worked example above. On made-up numbers: three data points, two features, and a layer with **three** units.

```python
P = np.array([[1.0, 0.0],
              [2.0, 3.0],
              [0.0, 1.0]])
A = np.array([[1.0, 0.0],     # unit 1's weights
              [0.0, 2.0],     # unit 2's weights
              [1.0, 1.0]])    # unit 3's weights
c = np.array([0.0, 1.0, -1.0])
print(P @ A.T + c)
# [[1. 1. 0.]
#  [2. 7. 4.]
#  [0. 3. 0.]]
```

Check the shapes: `P` is `(3, 2)` and `A.T` is `(2, 3)`, so the result is `(3, 3)`: one row per data point, one column per unit. The bias vector `c` is added to **every row**.

</details>

<details>
<summary>Hint for <code>H</code>: calling a function you were passed</summary>

Inside `forward`, `g` is an ordinary variable that holds a function, so you call it like any function: `g(...)`. A made-up example:

```python
def apply_twice(f, x):
    return f(f(x))

print(apply_twice(np.sqrt, 16.0))   # 2.0: sqrt(sqrt(16))
```

</details>

In [ ]:
# 2
def forward(X, W1, b1, W2, b2, g):
    "Output scores z2, one per row of X, of a network with one hidden layer (activation g)."
    # Z1: the hidden scores, one row per data point.  H: g applied to Z1.
    ### YOUR CODE HERE
    Z2 = H @ W2.T + b2        # the output layer (given): same pattern, one column per output unit
    return Z2[:, 0]           # the one output unit's scores, one per data point

In [ ]:
# Made-up weights. Unit 1's and unit 2's weights differ, so a missing .T gives different scores.
X_chk  = np.array([[0.2, 0.7], [1.0, -1.0]])
W1_chk = np.array([[1.0, 2.0], [-1.0, 0.5]])
b1_chk = np.array([0.1, -0.2])
W2_chk = np.array([[2.0, -1.0]])
b2_chk = np.array([0.3])
z_step_chk = forward(X_chk, W1_chk, b1_chk, W2_chk, b2_chk, step)
z_sig_chk  = forward(X_chk, W1_chk, b1_chk, W2_chk, b2_chk, sigmoid)
check('forward returns one score per data point', np.shape(z_step_chk) == (2,))
check('forward with step gives [2.3, 0.3]', np.allclose(z_step_chk, [2.3, 0.3]))
check('forward with sigmoid gives [1.504, 0.724]', np.allclose(z_sig_chk, [1.504, 0.724], atol=1e-3))

## Task 3: Test the XOR network

Think of each feature as **True** when it's near 1 and **False** when it's near 0. XOR is 1 when exactly one feature is True. The demo's XOR network ([Following the data through the network](ch_6_and_10_02_neurons_to_networks.ipynb#forward-pass)) has step hidden units:

- **Hidden unit 1:** weights $(1, 1)$, bias $-0.5$: "is at least one feature True?"
- **Hidden unit 2:** weights $(1, 1)$, bias $-1.5$: "are both True?"
- **Output unit:** weights $(1, -1)$, bias $-0.5$: "at least one, but not both."

**3 · Code.**

1. Define `X_test`, one row per test point. Include **at least one point near each of the four corners**, and **at least one that isn't exactly a corner**, like $(0.9, 0.2)$. Keep each feature within 0.2 of 0 or 1. Comment each row with its corner, like `# True, False`.
2. Run the step network on `X_test`: compute its output scores `z_step` with `forward`, and its predictions `pred_step` (1 when the score is positive) with `step`.


<details>
<summary>Hint for <code>X_test</code>: a commented array</summary>

Each row of a 2D array can go on its own line, with its own comment. A made-up example:

```python
grid = np.array([[0.0, 0.5],    # left edge, middle
                 [2.0, 2.0],    # top right
                 [1.0, 0.0]])   # bottom, middle
print(grid.shape)               # (3, 2)
```

</details>

<details>
<summary>Hint for <code>z_step</code> and <code>pred_step</code></summary>

`forward` takes `(X, W1, b1, W2, b2, g)`, in that order. Pass the activation function itself, by name, without calling it. Then `step` turns the scores into 0s and 1s.

</details>

In [ ]:
# 3
W1 = np.array([[1.0, 1.0],     # hidden unit 1's weights (given)
               [1.0, 1.0]])    # hidden unit 2's weights
b1 = np.array([-0.5, -1.5])
W2 = np.array([[1.0, -1.0]])   # the output unit's weights (one row: one output unit)
b2 = np.array([-0.5])

# X_test: one row per test point, each with a comment like  # True, False
### YOUR CODE HERE

# z_step: the step network's output scores.  pred_step: its predictions.
### YOUR CODE HERE


In [ ]:
y_test = (X_test.round().sum(axis=1) == 1).astype(float)   # the XOR label of each point's corner

print('point           prediction  label')
for x, p, t in zip(X_test, pred_step, y_test):
    print(f'({x[0]:.2f}, {x[1]:.2f})  {p:8.0f} {t:8.0f}')

near = np.all((np.abs(X_test) <= 0.2) | (np.abs(X_test - 1) <= 0.2))
corners = {tuple(r) for r in X_test.round()}
check('X_test has two features per point', X_test.ndim == 2 and X_test.shape[1] == 2)
check('every feature is within 0.2 of 0 or 1', near)
check('X_test covers all four corners', len(corners) == 4)
check('at least one point is not exactly a corner', np.any(X_test != X_test.round()))
check('the step network classifies every test point correctly', np.array_equal(pred_step, y_test))

## Task 4: The same network with sigmoid hidden units

Your `forward` takes the activation as an argument, so running the same network with sigmoid hidden units is one change: pass `sigmoid` instead of `step`. Keep the weights from Task 3.

**4 · Code + Written.**

1. Run the network on `X_test` with **sigmoid** hidden units: compute its output scores `z_sig` with `forward`, and its predictions `pred_sig` (1 when the score is positive) with `step`.
2. **Written:** it no longer classifies XOR. Why not? Use the hints. **Take your best guess:** we haven't covered this yet, so any reasonable attempt gets full credit, and we'll discuss it in class.


<details>
<summary>Hint for step 1</summary>

It's Task 3's two lines, with `sigmoid` in place of `step` as `forward`'s last argument. Keep `step` for the predictions.

</details>

<details>
<summary>Hint for step 2 (Written)</summary>

- Compare `pred_sig` with the labels `y_test`. Which points are wrong?
- Print the hidden outputs both ways: `step(X_test @ W1.T + b1)` and `sigmoid(X_test @ W1.T + b1)`. How close to 0 and 1 are the sigmoid's?
- The output unit computes $h_1 - h_2 - 0.5$. With step hidden units, what is $h_1 - h_2$ at the True, False corner? With sigmoid hidden units, how big does $h_1 - h_2$ get?

</details>

In [ ]:
# 4
### YOUR CODE HERE
print('scores:     ', z_sig.round(3))
print('predictions:', pred_sig)
print('labels:     ', y_test)

In [ ]:
check('z_sig has one score per test point', np.shape(z_sig) == (len(X_test),))
check('with sigmoid hidden units, the network misclassifies some test points', not np.array_equal(pred_sig, y_test))

**4 Answer:**

_YOUR ANSWER HERE_


## Optional: What if the hidden layer has no activation? (not graded)

The demo's section [Why the activation can't be left out](ch_6_and_10_02_neurons_to_networks.ipynb#activation-matters) showed that without an activation, the two layers multiply out into **one neuron**. In matrix form, $W_2(W_1\mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = (W_2W_1)\,\mathbf{x} + (W_2\mathbf{b}_1 + \mathbf{b}_2)$, so the one neuron's weights and bias are

$$
W_{\text{one}} = W_2\,W_1, \qquad \mathbf{b}_{\text{one}} = W_2\,\mathbf{b}_1 + \mathbf{b}_2.
$$

$W_{\text{one}}$ is $1 \times 2$: one row, for one neuron.

"No activation" means $g(z) = z$, which you can pass to `forward` as `lambda z: z` (a function that returns its input unchanged).

1. Run the step network's weights with no hidden activation: compute `z_none` and its predictions `pred_none`.
2. Compute `W_one` and `b_one` from the formulas above, and `z_one`, the single neuron's scores: the same pattern as `forward`'s output layer, `(X_test @ W_one.T + b_one)[:, 0]`. The check confirms that they match `z_none`.
3. **Written:** which points does the network get wrong? Look at `W_one` and `b_one`: why does every point get the same score?


<details>
<summary>Hint for step 2: from formulas to code</summary>

| Math | NumPy |
|---|---|
| matrix times matrix, $AB$ | `A @ B` |
| matrix times vector, $A\mathbf{v}$ | `A @ v` |

On made-up numbers:

```python
A = np.array([[1, -1]])         # 1 x 2
B = np.array([[1, 2],
              [3, 4]])
print(A @ B)                    # [[-2 -2]]: 1 x 2
print(A @ np.array([5, 7]))     # [-2]: 1*5 + (-1)*7
```

</details>

In [ ]:
# Optional
identity = lambda z: z          # "no activation"
### YOUR CODE HERE
print('scores, no activation:', z_none)
print('predictions:          ', pred_none)
print('W_one =', W_one, '  b_one =', b_one)

In [ ]:
check('the single neuron gives the same scores as the network', np.allclose(z_one, z_none))
check('without the activation, the network misclassifies some points', not np.array_equal(pred_none, y_test))

**Optional Answer:**

_YOUR ANSWER HERE_
